## **Etapa 1: Preparar e Selecionar os Dados**

In [ ]:
import pandas as pd

PicMoney_BaseTransacoes = "PicMoney_Transacao.csv"
PicMoney_BasePlayers = "PicMoney_Players.csv"
PicMoney_BaseAvPaulista = "PicMoney_Av__Paulista.csv"
PicMoney_BaseLojas = "PicMoney_Lojas_e_Valores.csv"

dfTransacoes = pd.read_csv(PicMoney_BaseTransacoes, sep=';')
dfPlayers = pd.read_csv(PicMoney_BasePlayers, sep=';')
dfPaulista = pd.read_csv(PicMoney_BaseAvPaulista, sep=';')
dfLojas = pd.read_csv(PicMoney_BaseLojas, sep=';')

print("Base Transações Bruta: \n", dfTransacoes.head())
print("\nBase Players Bruta: \n", dfPlayers.head())
print("\nBase Av. Paulista Bruta: \n", dfPaulista.head())
print("\nBase Lojas e Valores Bruta: \n", dfLojas.head())

colunasTransacoes = ["data","hora","bairro_estabelecimento","categoria_estabelecimento","tipo_cupom","valor_cupom","repasse_picmoney"]
colunasPlayers = ["idade", "sexo"]
colunasPedestreAvPaulista = ["possui_app_picmoney"]

dfSelTransacoes = dfTransacoes[colunasTransacoes].copy()
dfSelPlayers = dfPlayers[colunasPlayers].copy()
dfSelPesdestreAvPaulista = dfPaulista[colunasPedestreAvPaulista].copy()

print("\n\nDados que vamos utilizar na base PicMoney_BaseTransacoes: \n", dfSelTransacoes.head())
print("\n\nDados que vamos utilizar na base PicMoney_BasePlayers: \n", dfSelPlayers.head())
print("\n\nDados que vamos utilizar na base PicMoney_BaseLojas: \nNenhum, base redundante em relação as outras.")
print("\n\nDados que vamos utilizar na base PicMoney_BasePedestresAvPaulista: \n", dfSelPesdestreAvPaulista.head())

## **Etapa 2: Limpar/Uniformizar os Dados**

In [ ]:
#Essa etapa foi feita utilizando MySQL e ferramentas do excel para melhorar a legibilidade e a integração dos dados.
#As informações foram passadas no PDF.

## **Etapa 3: Derivar Dados**

In [ ]:
#Essa etapa eu fiz utilizando minhas bases corrigidas, irei deixar elas em uma pasta no github junto aos arquivos da segunda entrega
!pip install unidecode
import pandas as pd
import re
from unidecode import unidecode

PicMoney_BaseTransacoes_Corrigida = "PicMoney_Base_Transacoes_Corrigida.xlsx"
PicMoney_BasePlayers_Corrigida = "PicMoney_Base_Cadastral_de_Players_Corrigida.xlsx"
PicMoney_BaseAvPaulista_Corrigida = "PicMoney_Base_Pedestres_Av__Paulista_Corrigida.xlsx"
PicMoney_BaseLojas_Corrigida = "PicMoney_Lojas_e_Valores_Corrigida.xlsx"

dfTransacoesCorrigido = pd.read_excel(PicMoney_BaseTransacoes_Corrigida)
dfPlayersCorrigido = pd.read_excel(PicMoney_BasePlayers_Corrigida)
dfAvPaulistaCorrigido = pd.read_excel(PicMoney_BaseAvPaulista_Corrigida)
dfLojasCorrigido = pd.read_excel(PicMoney_BaseLojas_Corrigida)

#Derivando 'data_hora_transação'
data_str = dfTransacoesCorrigido['data'].dt.strftime('%d/%m/%Y')
hora_str = dfTransacoesCorrigido['hora'].astype(str)
dfTransacoesCorrigido['data_hora_str'] = data_str + ' ' + hora_str

dfTransacoesCorrigido['data_hora_transacao'] = pd.to_datetime(
    dfTransacoesCorrigido['data_hora_str'],
    format='%d/%m/%Y %H:%M:%S',
    errors='coerce'
    )

linhas_antes = len(dfTransacoesCorrigido)
dfTransacoesCorrigido = dfTransacoesCorrigido.dropna(subset=['data_hora_transacao'])
linhas_depois = len(dfTransacoesCorrigido)

dfTransacoesCorrigido = dfTransacoesCorrigido.drop(columns=['data', 'hora', 'data_hora_str'])

print("Coluna 'data_hora_transacao': ")
print(dfTransacoesCorrigido[['data_hora_transacao']].head())

In [23]:
#Derivando 'Faixa-Etária'
faixas = [0, 18, 25, 35, 45, 55, 65, 100]
rotulos = ['0-18', '19-25', '26-35', '36-45', '46-55', '56-65', '65+']

dfPlayersCorrigido['idade'] = pd.to_numeric(dfPlayersCorrigido['idade'], errors='coerce')

dfPlayersCorrigido['faixa_etaria'] = pd.cut(
    dfPlayersCorrigido['idade'],
    bins=faixas,
    labels=rotulos,
    right=False
)

print("Coluna 'faixa_etaria':")
print(dfPlayersCorrigido[['idade', 'faixa_etaria']].head())

Coluna 'faixa_etaria':
   idade faixa_etaria
0     55        56-65
1     44        36-45
2     45        46-55
3     68          65+
4     55        56-65


In [10]:
#margem individual
dfTransacoesCorrigido['margem'] = (
    (dfTransacoesCorrigido['repasse_picmoney'] / dfTransacoesCorrigido['valor_cupom']) * 100
).round(2)
print(dfTransacoesCorrigido[['repasse_picmoney', 'valor_cupom', 'margem']].head())

#margem total
total_repasse = dfTransacoesCorrigido['repasse_picmoney'].sum()
total_valor = dfTransacoesCorrigido['valor_cupom'].sum()
margem_total = (total_repasse / total_valor) * 100
margem_total = round(margem_total, 2)

print(f"\n\nMargem Total: {margem_total}%")

   repasse_picmoney  valor_cupom  margem
0             11.48       229.64    5.00
1             17.82       356.33    5.00
2             27.61       719.06    3.84
3             25.85       798.34    3.24
4             28.85       718.45    4.02


Margem Total: 12.8%


In [13]:
#Receita Líquida por Tipo de Cupom
df_receita_cupom = dfTransacoesCorrigido.groupby('tipo_cupom')['repasse_picmoney'].sum().reset_index()
df_receita_cupom.columns = ['tipo_cupom', 'receita_liquida']

print("Receita por Tipo de Cupom:\n")
print(df_receita_cupom)

Receita por Tipo de Cupom:

  tipo_cupom  receita_liquida
0   Cashback        924185.68
1   Desconto       5480485.92
2    Produto        642802.31


In [21]:
#TOP5 Parceiros
df_parceiros = dfTransacoesCorrigido.groupby('nome_estabelecimento')['repasse_picmoney'].sum().reset_index()
df_parceiros = df_parceiros.sort_values(by='repasse_picmoney', ascending=False)
df_parceiros_grafico = df_parceiros.head(5)

print("TOP5 Parceiros:\n")
print(df_parceiros_grafico)

TOP10 Parceiros:

   nome_estabelecimento  repasse_picmoney
8   Drogaria SÃ£o Paulo         303757.25
9              Drogasil         300424.52
7            Droga Raia         298291.31
15            Lavoisier         234265.52
26                Sabin         229908.12


In [17]:
#Receita por bairro
df_bairro = dfTransacoesCorrigido.groupby('bairro_estabelecimento')['repasse_picmoney'].sum().reset_index()
df_bairro.columns = ['bairro', 'receita_liquida']

print("Receita por Bairro:\n")
print(df_bairro)

Receita por Bairro:

           bairro  receita_liquida
0      Bela Vista        328446.99
1        ButantÃ£        341146.21
2     Campo Limpo        340037.82
3      Casa Verde        331814.60
4    ConsolaÃ§Ã£o        337628.23
5   HigienÃ³polis        334216.91
6        Ipiranga        343124.81
7        Itaquera        329934.08
8       Jabaquara        337112.51
9            Lapa        342744.22
10         LimÃ£o        343275.01
11          Penha        329710.65
12       Perdizes        334956.74
13      Pinheiros        331755.40
14     RepÃºblica        335099.06
15        Santana        336494.39
16    Santo Amaro        326083.61
17            SÃ©        337080.17
18       TatuapÃ©        337590.01
19       Tucuruvi        327174.56
20  Vila Prudente        342047.93


In [19]:
#Taxa de Retenção
dfTransacoesCorrigido['data_hora_transacao'] = pd.to_datetime(dfTransacoesCorrigido['data_hora_transacao'], errors='coerce')
inicio = dfTransacoesCorrigido['data_hora_transacao'].min()
fim_primeira_semana = inicio + pd.Timedelta(days=7)

inicio_quarta_semana = inicio + pd.Timedelta(days=21)
fim_quarta_semana = inicio + pd.Timedelta(days=28)

usuarios_primeira_semana = set(
    dfTransacoesCorrigido[
        (dfTransacoesCorrigido['data_hora_transacao'] >= inicio) &
        (dfTransacoesCorrigido['data_hora_transacao'] < fim_primeira_semana)
    ]['celular']
)

usuarios_quarta_semana = set(
    dfTransacoesCorrigido[
        (dfTransacoesCorrigido['data_hora_transacao'] >= inicio_quarta_semana) &
        (dfTransacoesCorrigido['data_hora_transacao'] < fim_quarta_semana)
    ]['celular']
)

usuarios_retenidos = usuarios_primeira_semana.intersection(usuarios_quarta_semana)
taxa_retencao = (len(usuarios_retenidos) / len(usuarios_primeira_semana)) * 100 if len(usuarios_primeira_semana) > 0 else 0

print(f"Taxa de Retenção entre 1ª semana e 4ª semana: {taxa_retencao:.2f}%")

Taxa de Retenção entre 1ª semana e 4ª semana: 94.77%


In [20]:
#Dia e Horário de Pico
dfTransacoesCorrigido['data_hora_transacao'] = pd.to_datetime(dfTransacoesCorrigido['data_hora_transacao'], errors='coerce')
dfTransacoesCorrigido['hora_transacao'] = dfTransacoesCorrigido['data_hora_transacao'].dt.hour
dfTransacoesCorrigido['dia_semana'] = dfTransacoesCorrigido['data_hora_transacao'].dt.day_name()

hora_de_pico = dfTransacoesCorrigido['hora_transacao'].mode()[0]
dia_de_pico = dfTransacoesCorrigido['dia_semana'].mode()[0]

print(f"Horário de Pico: {hora_de_pico}:00")
print(f"Dia de Pico: {dia_de_pico}")

Hora de Pico: 16:00
Dia de Pico: Thursday


## **Integrar os Dados**

In [28]:
#Integração de dados - tabela transação e players
#Necessário rodar celular de derivação
if 'celular' in dfTransacoesCorrigido.columns:
    dfTransacoesCorrigido = dfTransacoesCorrigido.rename(columns={'celular': 'celular_player'})
if 'celular' in dfPlayersCorrigido.columns:
    dfPlayersCorrigido = dfPlayersCorrigido.rename(columns={'celular': 'celular_player'})
if 'sexo' in dfPlayersCorrigido.columns:
    dfPlayersCorrigido = dfPlayersCorrigido.rename(columns={'sexo': 'genero'})

colunas_players_para_merge = [
    'celular_player',
    'idade',
    'genero',
    'faixa_etaria'
]
df_players_para_merge = dfPlayersCorrigido[colunas_players_para_merge].drop_duplicates(subset=['celular_player'])

df_final_integrado = pd.merge(
    dfTransacoesCorrigido,
    df_players_para_merge,
    on='celular_player',
    how='left'
)

print("\nDataFrame final (integrando a tabela Transação e Players:")
print(df_final_integrado[[
    'data_hora_transacao', 'celular_player', 'nome_estabelecimento',
    'repasse_picmoney', 'idade', 'genero', 'faixa_etaria'
]].head())


DataFrame final (integrando a tabela Transação e Players:
  data_hora_transacao   celular_player nome_estabelecimento  repasse_picmoney  \
0 2025-07-10 16:15:00  (61) 96497-8673              Habib's             11.48   
1 2025-07-15 08:15:00  (11) 94231-6424            Smart Fit             17.82   
2 2025-07-20 16:45:00  (11) 97965-2178              Outback             27.61   
3 2025-07-20 15:45:00  (11) 93418-4646               Subway             25.85   
4 2025-07-07 11:00:00  (11) 97973-1725        Octavio CafÃ©             28.85   

   idade     genero faixa_etaria  
0     30      Outro        26-35  
1     77  Masculino          65+  
2     55  Masculino        56-65  
3     78   Feminino          65+  
4     78  Masculino          65+  


## **Formatar Dados**

In [32]:
#Formatando os dados
df_final_integrado['valor_cupom'] = pd.to_numeric(df_final_integrado['valor_cupom'], errors='coerce').fillna(0)
df_final_integrado['repasse_picmoney'] = pd.to_numeric(df_final_integrado['repasse_picmoney'], errors='coerce').fillna(0)
df_final_integrado['idade'] = pd.to_numeric(df_final_integrado['idade'], errors='coerce')

print("Tipos de dados (colunas Transação e Players):")
print(df_final_integrado[['valor_cupom', 'repasse_picmoney', 'idade', 'data_hora_transacao']].dtypes)

print("\nFormatação de exibição de dados:")

def formatar_brl(valor):
    try:
        texto_padrao = f"{valor:,.2f}"
        texto_temp = texto_padrao.replace(",", "X")
        texto_br = texto_temp.replace(".", ",")
        texto_final = texto_br.replace("X", ".")
        return f"R$ {texto_final}"
    except Exception:
        return "R$ 0,00"

def formatar_numero_compacto(valor):
    try:
        valor = float(valor)
        if valor >= 1_000_000:
            return f"{valor / 1_000_000:.1f}M"
        if valor >= 1_000:
            return f"{valor / 1_000:.1f}K"
        return f"{valor:,.0f}"
    except Exception:
        return "0"

exemplo_receita = df_final_integrado['repasse_picmoney'].sum()
exemplo_usuarios = df_final_integrado['celular_player'].nunique()

print(f"Formatação de Moeda: {formatar_brl(exemplo_receita)}")
print(f"Formatação Compacta: {formatar_numero_compacto(exemplo_usuarios)}")

Tipos de dados (colunas Transação e Players):
valor_cupom                   float64
repasse_picmoney              float64
idade                           int64
data_hora_transacao    datetime64[ns]
dtype: object

Formatação de exibição de dados:
Formatação de Moeda: R$ 7.047.473,91
Formatação Compacta: 4.8K
